## 🎯 Objective

The purpose of this notebook is to **understand the structure and quality of the dataset** stored in the database and assess the need for **aggregated tables** that can support key business goals, including:

- 📈 **Vendor Selection for Profitability**:  
  Analyze vendor performance metrics to identify which vendors contribute most effectively to profitability, and support informed vendor selection strategies.

- 💰 **Product Pricing Optimization**:  
  Explore sales, cost, and product-related data to identify opportunities for optimizing product pricing that maximizes margin without impacting demand.

This initial exploration will provide insights into:
- The schema and distribution of available data.
- Potential data quality issues or missing values.
- Aggregation or transformation needs to power downstream decision-making and analytics.

---


In [20]:
import sys
from pathlib import Path

# Append src directory to sys.path
sys.path.append(str(Path("..", "src").resolve()))

In [ ]:
import pandas as pd
from infrastructure.postgres_connection import get_sql_database
from sqlalchemy import text

In [22]:
engine = get_sql_database()

✅ Successfully connected to the PostgreSQL cloud database.


In [24]:
tables_df = pd.read_sql(
    """
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
    AND table_schema NOT IN ('pg_catalog', 'information_schema');
    """,
    con=engine
)
print(tables_df)

  table_schema       table_name
0    my_schema  begin_inventory
1    my_schema    end_inventory
2    my_schema  purchase_prices


In [30]:
schema_name = "my_schema"

# Loop through each table and get row count and sample data
for table_name in tables_df['table_name']:
    # Schema-qualified table name
    full_table_name = f'"{schema_name}"."{table_name}"'

    # Get row count
    count_query = text(f'SELECT COUNT(*) as count FROM {full_table_name}')
    count_df = pd.read_sql(count_query, con=engine)
    count = count_df['count'].iloc[0]
    print(f"📊 Table '{table_name}' has {count} records.")

    # Get preview
    preview_query = text(f'SELECT * FROM {full_table_name} LIMIT 5')
    preview_df = pd.read_sql(preview_query, con=engine)
    display(preview_df)

📊 Table 'begin_inventory' has 206529 records.


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,startDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,8,12.99,2024-01-01
1,1_HARDERSFIELD_60,1,HARDERSFIELD,60,Canadian Club 1858 VAP,750mL,7,10.99,2024-01-01
2,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,6,36.99,2024-01-01
3,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,3,38.99,2024-01-01
4,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,6,34.99,2024-01-01


📊 Table 'end_inventory' has 224489 records.


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,endDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,11,12.99,2024-12-31
1,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,7,36.99,2024-12-31
2,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,7,38.99,2024-12-31
3,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,4,34.99,2024-12-31
4,1_HARDERSFIELD_75,1,HARDERSFIELD,75,Three Olives Tomato Vodka,750mL,7,14.99,2024-12-31


📊 Table 'purchase_prices' has 12261 records.


,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,58,Gekkeikan Black & Gold Sake,12.99,750mL,750,1,9.28,8320,SHAW ROSS INT L IMP LTD
1,62,Herradura Silver Tequila,36.99,750mL,750,1,28.67,1128,BROWN-FORMAN CORP
2,63,Herradura Reposado Tequila,38.99,750mL,750,1,30.46,1128,BROWN-FORMAN CORP
3,72,No. 3 London Dry Gin,34.99,750mL,750,1,26.11,9165,ULTRA BEVERAGE COMPANY LLP
4,75,Three Olives Tomato Vodka,14.99,750mL,750,1,10.94,7245,PROXIMO SPIRITS INC.


### Observation from the table view
- The `begin_inventory` and `end_inventory` tables contain inventory data at the start and end of the year, which is not relevant for analyzing vendor behavior.
- Therefore, these tables can be excluded from the analysis.

We will start to examine how vendor-related data is distributed across other tables.

In [6]:
# Select one vendor from the dataset for analysis
vendor_no = 4466

### Puchases Table 

In [32]:
query = text('SELECT * FROM my_schema.purchases WHERE "VendorNumber" = :vendor_no')
purchases = pd.read_sql(query, con=engine, params={"vendor_no": vendor_no})

In [33]:
purchases.head()

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
1,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
2,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
3,38_GOULCREST_5215,38,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,2023-12-27,2024-01-07,2024-01-19,2024-02-26,9.41,6,56.46,1
4,59_CLAETHORPES_5215,59,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,2023-12-27,2024-01-05,2024-01-19,2024-02-26,9.41,6,56.46,1


### Puchase prices

In [34]:
query = text("""
    SELECT * FROM my_schema.purchase_prices
    WHERE "VendorNumber" = :vendor_no
""")
purchase_prices = pd.read_sql(query, con=engine, params={"vendor_no": vendor_no})

In [35]:
purchase_prices.head()

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,5215,TGI Fridays Long Island Iced,12.99,1750mL,1750,1,9.41,4466,AMERICAN VINTAGE BEVERAGE
1,5255,TGI Fridays Ultimte Mudslide,12.99,1750mL,1750,1,9.35,4466,AMERICAN VINTAGE BEVERAGE
2,3140,TGI Fridays Orange Dream,14.99,1750mL,1750,1,11.19,4466,AMERICAN VINTAGE BEVERAGE


### Vendor Invoice

In [ ]:
query = text("""
    SELECT * FROM my_schema.vendor_invoice
    WHERE "VendorNumber" = :vendor_no
""")
vendor_invoice = pd.read_sql(query, con=engine, params={"vendor_no": vendor_no})

In [12]:
vendor_invoice.head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-19,8207,2023-12-27,2024-02-26,335,3142.33,16.97,None
2,4466,AMERICAN VINTAGE BEVERAGE,2024-01-18,8307,2024-01-03,2024-02-18,41,383.35,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,2024-01-27,8469,2024-01-14,2024-03-11,72,673.20,3.30,None
4,4466,AMERICAN VINTAGE BEVERAGE,2024-02-04,8532,2024-01-19,2024-03-15,79,740.21,3.48,None


### Sales

In [ ]:
query = text("""
    SELECT * FROM my_schema.sales
    WHERE "VendorNo" = :vendor_no
""")
sales = pd.read_sql(query, con=engine, params={"vendor_no": vendor_no})

In [14]:
sales.head()

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
1,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-12,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
2,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-15,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
3,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
4,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE


### Observation 
Due to the huge number of rows in `Sales` table, query time is quite long. We will need to optimised any query related to `sales` table

In [15]:
purchases.groupby(["Brand","PurchasePrice"])[["Quantity","Dollars"]].sum()

,,Quantity,Dollars
Brand,PurchasePrice,,
3140,11.19,4640,51921.60
5215,9.41,4923,46325.43
5255,9.35,6215,58110.25


In [16]:
purchases.groupby(["PONumber"])[["Quantity","Dollars"]].sum()

,Quantity,Dollars
PONumber,,
8137,15,140.55
8207,335,3142.33
8307,41,383.35
8469,72,673.20
8532,79,740.21
8604,347,3261.37
8793,72,675.36
8892,117,1096.05
8995,129,1209.27


In [17]:
sales.groupby("Brand")[['SalesQuantity','SalesDollars','SalesPrice','ExciseTax']].sum()

,SalesQuantity,SalesDollars,SalesPrice,ExciseTax
Brand,,,,
3140,3890,50531.10,30071.85,7149.25
5215,4651,60416.49,41542.02,8548.96
5255,6096,79187.04,51180.60,11204.28


## 📋 Dataset Overview

- The **purchases** table contains actual purchase records, including:
  - Date of purchase
  - Products (brands) purchased by vendors
  - Amount paid (in dollars)
  - Quantity purchased

- The **purchase_prices** table provides product-wise pricing details, including:
  - Actual prices and purchase prices for each product
  - Uniqueness of records based on the combination of vendor and brand

- The **vendor_invoice** table aggregates data from the purchases table, summarizing:
  - Quantity and dollar amounts
  - Freight costs per vendor
  - Uniqueness maintained by vendor and purchase order (PO) number

- The **sales** table captures actual sales transactions with details on:
  - Brands purchased by vendors
  - Quantity sold
  - Selling price and revenue earned

---

### 🔍 Need for a Summary Table

Since the data required for analysis is spread across multiple tables, it is essential to create a **summary table** that consolidates:

- Purchase transactions made by vendors
- Sales transaction data
- Freight costs associated with each vendor
- Actual product prices offered by vendors

This summary table will enable more efficient and accurate analysis to support vendor selection and product pricing optimization.


In [ ]:
## Explore each summary 

freight_summary = pd.read_sql("""
    SELECT 
        "VendorNumber", 
        SUM("Freight") AS "FreightCost" 
    FROM my_schema.vendor_invoice 
    GROUP BY "VendorNumber"
""", con=engine)


,VendorNumber,FreightCost
0,2,27.08
1,54,0.48
2,60,367.52
3,105,62.39
4,200,6.19
...,...,...
121,98450,856.02
122,99166,130.09
123,172662,178.34
124,173357,202.50


In [ ]:
purchase_summary = pd.read_sql("""
    SELECT 
        p."VendorNumber",
        p."VendorName",
        p."Brand",
        p."Description",
        p."PurchasePrice",
        pp."Price" AS "ActualPrice",
        pp."Volume",
        SUM(p."Quantity") AS "TotalPurchaseQuantity",
        SUM(p."Dollars") AS "TotalPurchaseDollars"
    FROM my_schema.purchases p
    JOIN my_schema.purchase_prices pp
        ON p."Brand" = pp."Brand"
    WHERE p."PurchasePrice" > 0
    GROUP BY 
        p."VendorNumber", p."VendorName", 
        p."Brand", p."Description", 
        p."PurchasePrice", pp."Price", pp."Volume"
    ORDER BY "TotalPurchaseDollars"
""", con=engine)

KeyboardInterrupt: 

In [ ]:
sales_summary = pd.read_sql("""
    SELECT 
        "VendorNo",
        "Brand",
        SUM("SalesQuantity") AS "TotalSalesQuantity",
        SUM("SalesDollars") AS "TotalSalesDollars",
        SUM("SalesPrice") AS "TotalSalesPrice",
        SUM("ExciseTax") AS "TotalExciseTax"
    FROM my_schema.sales
    GROUP BY "VendorNo", "Brand"
""", con=engine)

### Merge all the summary table to create an aggregated summary table for further analysis

In [ ]:
vendor_sales_summary = pd.read_sql("""
WITH FreightSummary AS (
    SELECT 
        "VendorNumber", 
        SUM("Freight") AS "FreightCost"
    FROM my_schema.vendor_invoice
    GROUP BY "VendorNumber"
), 

PurchaseSummary AS (
    SELECT 
        p."VendorNumber",
        p."VendorName",
        p."Brand",
        p."Description",
        p."PurchasePrice",
        pp."Price" AS "ActualPrice",
        pp."Volume",
        SUM(p."Quantity") AS "TotalPurchaseQuantity",
        SUM(p."Dollars") AS "TotalPurchaseDollars"
    FROM my_schema.purchases p
    JOIN my_schema.purchase_prices pp
        ON p."Brand" = pp."Brand"
    WHERE p."PurchasePrice" > 0
    GROUP BY 
        p."VendorNumber", p."VendorName", p."Brand", p."Description", 
        p."PurchasePrice", pp."Price", pp."Volume"
), 

SalesSummary AS (
    SELECT 
        "VendorNo",
        "Brand",
        SUM("SalesQuantity") AS "TotalSalesQuantity",
        SUM("SalesDollars") AS "TotalSalesDollars",
        SUM("SalesPrice") AS "TotalSalesPrice",
        SUM("ExciseTax") AS "TotalExciseTax"
    FROM my_schema.sales
    GROUP BY "VendorNo", "Brand"
) 

SELECT 
    ps."VendorNumber",
    ps."VendorName",
    ps."Brand",
    ps."Description",
    ps."PurchasePrice",
    ps."ActualPrice",
    ps."Volume",
    ps."TotalPurchaseQuantity",
    ps."TotalPurchaseDollars",
    ss."TotalSalesQuantity",
    ss."TotalSalesDollars",
    ss."TotalSalesPrice",
    ss."TotalExciseTax",
    fs."FreightCost"
FROM PurchaseSummary ps
LEFT JOIN SalesSummary ss 
    ON ps."VendorNumber" = ss."VendorNo" 
    AND ps."Brand" = ss."Brand"
LEFT JOIN FreightSummary fs 
    ON ps."VendorNumber" = fs."VendorNumber"
ORDER BY ps."TotalPurchaseDollars" DESC
""", con=engine)


## 📝 Vendor-wise Sales and Purchase Summary Query

This query generates a comprehensive vendor-wise summary of sales and purchases, which is critical for:

### Performance Optimization
- The query involves complex joins and aggregations over large datasets such as sales and purchases.
- Pre-aggregating and storing results helps avoid repeated expensive computations.
- Enables detailed analysis of sales, purchases, and pricing across different vendors and brands.
- Provides significant performance benefits for dashboarding and reporting by reducing query runtime.
- Dashboards can quickly fetch data from the pre-computed `vendor_sales_summary` table instead of running costly queries on raw data every time.

---

### Data Quality Assurance

Before using this summary for analysis, it is essential to:

- Clean the data to handle any inconsistencies or anomalies.
- Ensure data integrity to maintain accuracy in insights derived from the summary.
- This step guarantees reliable vendor performance evaluation and informed decision-making.


### Explore data and Data Cleaning 

In [ ]:
# Check columns names
vendor_sales_summary.columns

In [ ]:
# check data types
vendor_sales_summary.dtypes

In [ ]:
# check for missing values
vendor_sales_summary.isnull().sum()

In [ ]:
# check for unique volumes
vendor_sales_summary['Volume'].unique()

In [ ]:
# Check for unique vendors
vendor_sales_summary['VendorName'].unique()

### Feature Engineering

In [ ]:
# creating new columns for better analysis
vendor_sales_summary['GrossProfit'] = vendor_sales_summary['TotalSalesDollars'] - vendor_sales_summary['TotalPurchaseDollars']
vendor_sales_summary['ProfitMargin'] = (vendor_sales_summary['GrossProfit'] / vendor_sales_summary['TotalSalesDollars'])*100
vendor_sales_summary['StockTurnover'] = vendor_sales_summary['TotalSalesQuantity'] / vendor_sales_summary['TotalPurchaseQuantity']
vendor_sales_summary['SalesToPurchaseRatio'] = vendor_sales_summary['TotalSalesDollars'] / vendor_sales_summary['TotalPurchaseDollars']

In [ ]:
# changing datatype to float
vendor_sales_summary['Volume'] = vendor_sales_summary['Volume'].astype('float')

In [ ]:
# filling missing value with 0
vendor_sales_summary.fillna(0,inplace = True)

In [ ]:
# removing spaces from categorical columns
vendor_sales_summary['VendorName'] = vendor_sales_summary['VendorName'].str.strip()
vendor_sales_summary['Description'] = vendor_sales_summary['Description'].str.strip()

## We can save this data as a new table into the DataBase so we can save time by utilising optimised tables